In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-addon-aqc-tensor qiskit-addon-utils scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Bermula dengan Pengkompilan Kuantum Anggaran menggunakan rangkaian tensor (AQC-Tensor)


<details>
<summary><b>Versi pakej</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan menggunakan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
qiskit-addon-utils~=0.3.0
qiskit-addon-aqc-tensor[aer,quimb-jax]~=0.2.0; sys.platform != 'darwin'
scipy~=1.16.3
```
</details>
Panduan ini menunjukkan contoh kerja mudah untuk bermula dengan AQC-Tensor. Dalam contoh ini, anda akan mengambil sebuah Circuit Trotter yang mensimulasikan evolusi model Ising medan melintang dan menggunakan kaedah AQC-Tensor untuk mengurangkan kedalaman Circuit yang terhasil. Selain itu, contoh ini memerlukan pakej `qiskit-addon-utils` untuk penjana masalah, `qiskit-aer` untuk simulasi rangkaian tensor, dan `scipy` untuk pengoptimuman parameter.

Untuk bermula, ingat semula bahawa Hamiltonian model Ising medan melintang mempunyai bentuk

$$ \mathcal{H}_{Ising} = \sum_{i=1}^N J_{i,(i+1)}Z_iZ_{i+1} + h_i X_i $$

di mana kita andaikan syarat sempadan berkala yang bermakna untuk $i=10$ kita mendapat $i+1=11\rightarrow 1$ dan $J$ adalah kekuatan gandingan antara dua tapak dan $h$ adalah kekuatan medan magnet luaran.

Coretan kod berikut akan menjana Hamiltonian rantai Ising 10 tapak dengan syarat sempadan berkala.

In [1]:
from qiskit.transpiler import CouplingMap
from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian
from qiskit.synthesis import SuzukiTrotter
from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_aqc_tensor import generate_ansatz_from_circuit
from qiskit_addon_aqc_tensor.simulation import tensornetwork_from_circuit
from qiskit_addon_aqc_tensor.simulation import compute_overlap
from qiskit_addon_aqc_tensor.objective import MaximizeStateFidelity
from qiskit_aer import AerSimulator
from scipy.optimize import OptimizeResult, minimize


# Generate some coupling map to use for this example
coupling_map = CouplingMap.from_heavy_hex(3, bidirectional=False)

# Choose a 10-qubit circle on this coupling map
reduced_coupling_map = coupling_map.reduce(
    [0, 13, 1, 14, 10, 16, 4, 15, 3, 9]
)

# Get a qubit operator describing the Ising field model
hamiltonian = generate_xyz_hamiltonian(
    reduced_coupling_map,
    coupling_constants=(0.0, 0.0, 1.0),
    ext_magnetic_field=(0.4, 0.0, 0.0),
)

## Bahagikan evolusi masa kepada dua bahagian untuk pelaksanaan klasik dan kuantum
Matlamat keseluruhan contoh ini adalah untuk mensimulasikan evolusi masa Hamiltonian model. Kami melakukannya di sini melalui evolusi Trotter, yang akan dibahagikan kepada dua bahagian.

1. Bahagian awal yang boleh disimulasikan menggunakan keadaan hasil darab matrik (MPS). Inilah yang akan 'dikompil' menggunakan AQC-Tensor.
2. Bahagian seterusnya yang akan dilaksanakan pada perkakasan kuantum.

Kita akan pilih untuk mengevolusi sistem sehingga masa $t_f=5$ dan gunakan AQC-Tensor untuk memampatkan evolusi masa sehingga $t=4$, kemudian evolusi menggunakan langkah Trotter biasa sehingga $t_f$.

Dari sini kita akan menjana dua Circuit seterusnya, satu yang akan dimampatkan menggunakan AQC-Tensor dan satu lagi yang akan dilaksanakan pada QPU. Untuk Circuit pertama, memandangkan ia akan disimulasikan secara klasik menggunakan keadaan hasil darab matrik, kita akan menggunakan bilangan lapisan yang banyak untuk meminimumkan ralat Trotter. Sementara itu Circuit kedua yang mensimulasikan evolusi masa dari $t_i=4$ ke $t_f=5$ akan menggunakan lebih sedikit lapisan bagi meminimumkan kedalaman.

In [2]:
# Generate circuit to be compressed
aqc_evolution_time = 4.0
aqc_target_num_trotter_steps = 45

aqc_target_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=aqc_target_num_trotter_steps),
    time=aqc_evolution_time,
)

# Generate circuit to execute on hardware
subsequent_evolution_time = 1.0
subsequent_num_trotter_steps = 5

subsequent_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=subsequent_num_trotter_steps),
    time=subsequent_evolution_time,
)

Untuk tujuan perbandingan, kita juga akan menjana Circuit ketiga. Satu yang berevolusi sehingga $t=4$, tetapi mempunyai bilangan lapisan yang sama seperti Circuit kedua yang berevolusi dari $t_i=4$ ke $t_f=5$. Inilah Circuit yang *akan* kita laksanakan sekiranya kita tidak menggunakan teknik AQC-Tensor. Kita akan panggil ini Circuit *perbandingan* buat masa ini.

In [3]:
aqc_comparison_num_trotter_steps = int(
    subsequent_num_trotter_steps
    / subsequent_evolution_time
    * aqc_evolution_time
)
aqc_comparison_num_trotter_steps

comparison_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=aqc_comparison_num_trotter_steps),
    time=aqc_evolution_time,
)

## Jana ansatz dan bina simulasi MPS
Seterusnya kita akan menjana ansatz yang akan dioptimumkan. Ia akan berevolusi ke masa evolusi yang sama seperti Circuit pertama kita (dari $t_i=0$ ke $t_f=4$), tetapi dengan lebih sedikit langkah Trotter.

Setelah Circuit dijana, kita kemudian menghantar ia ke fungsi `generate_ansatz_from_circuit()` AQC-Tensor yang menganalisis kesambungan dua-Qubit dan mengembalikan dua perkara. Pertama adalah Circuit ansatz yang dijana dengan kesambungan dua-Qubit yang sama, dan kedua adalah satu set parameter yang, apabila dimasukkan ke dalam ansatz, menghasilkan Circuit input.

In [4]:
aqc_ansatz_num_trotter_steps = 5

aqc_good_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=aqc_ansatz_num_trotter_steps),
    time=aqc_evolution_time,
)

aqc_ansatz, aqc_initial_parameters = generate_ansatz_from_circuit(
    aqc_good_circuit, qubits_initially_zero=True
)
aqc_ansatz.draw("mpl", fold=-1)

<Image src="../docs/images/guides/qiskit-addons-aqc-get-started/extracted-outputs/e08edb92-da1f-4131-85f5-f89006f7a2dd-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-aqc-get-started/extracted-outputs/e08edb92-da1f-4131-85f5-f89006f7a2dd-0.svg)

Seterusnya kita akan membina representasi MPS bagi keadaan yang hendak dianggarkan oleh AQC. Kita juga akan mengira kesetiaan $|\langle\psi_1 | \psi_2 \rangle |^2$ antara keadaan yang disediakan oleh Circuit perbandingan berbanding Circuit yang menjana keadaan sasaran (yang menggunakan lebih banyak langkah Trotter).

In [5]:
# Generate MPS simulator settings and obtain the MPS representation of the target state
simulator_settings = AerSimulator(
    method="matrix_product_state",
    matrix_product_state_max_bond_dimension=100,
)
aqc_target_mps = tensornetwork_from_circuit(
    aqc_target_circuit, simulator_settings
)


# Compute the fidelity between the MPS representation of the target state and the state produced by the comparison circuit
comparison_mps = tensornetwork_from_circuit(
    comparison_circuit, simulator_settings
)
comparison_fidelity = (
    abs(compute_overlap(comparison_mps, aqc_target_mps)) ** 2
)
print(f"Comparison fidelity: {comparison_fidelity}")

Comparison fidelity: 0.9997111919739693


## Optimize the parameters of the ansatz using the MPS

Lastly, we will optimize the ansatz circuit such that it produces the target state better than our `comparison_fidelity`. The cost function to minimize will be the `MaximizeStateFidelity` and will be optimized using scipy's L-BFGS optimizer.

In [6]:
objective = MaximizeStateFidelity(
    aqc_target_mps, aqc_ansatz, simulator_settings
)

stopping_point = 1 - comparison_fidelity


def callback(intermediate_result: OptimizeResult):
    print(f"Intermediate result: Fidelity {1 - intermediate_result.fun:.8}")
    if intermediate_result.fun < stopping_point:
        # Good enough for now
        raise StopIteration


result = minimize(
    objective,
    aqc_initial_parameters,
    method="L-BFGS-B",
    jac=True,
    options={"maxiter": 100},
    callback=callback,
)
if (
    result.status
    not in (
        0,
        1,
        99,
    )
):  # 0 => success; 1 => max iterations reached; 99 => early termination via StopIteration
    raise RuntimeError(
        f"Optimization failed: {result.message} (status={result.status})"
    )

print(f"Done after {result.nit} iterations.")
aqc_final_parameters = result.x

Intermediate result: Fidelity 0.95084365


Intermediate result: Fidelity 0.98409893


Intermediate result: Fidelity 0.99142033


Intermediate result: Fidelity 0.99521405


Intermediate result: Fidelity 0.99566673


...

Intermediate result: Fidelity 0.99956469


Intermediate result: Fidelity 0.99964488


Intermediate result: Fidelity 0.99967419


Intermediate result: Fidelity 0.99968821


Intermediate result: Fidelity 0.9997448
Done after 24 iterations.


## Optimumkan parameter ansatz menggunakan MPS
Akhir sekali, kita akan mengoptimumkan Circuit ansatz supaya ia menghasilkan keadaan sasaran dengan kesetiaan yang lebih tinggi daripada `comparison_fidelity` kita. Fungsi kos untuk diminimumkan ialah `MaximizeStateFidelity` dan akan dioptimumkan menggunakan pengoptimum L-BFGS scipy.

In [7]:
final_circuit = aqc_ansatz.assign_parameters(aqc_final_parameters)
final_circuit.compose(subsequent_circuit, inplace=True)
final_circuit.draw("mpl", fold=-1)

<Image src="../docs/images/guides/qiskit-addons-aqc-get-started/extracted-outputs/45abbabe-0289-4a09-aa99-89f70bdc535d-0.svg" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Approximate quantum compilation for time evolution circuits](/docs/tutorials/approximate-quantum-compilation-for-time-evolution) tutorial.
</Admonition>